# ДЗ №3. Обработка потоков данных
### Выполнил: Лёвин В.А.
### Группа: ИУ6-54Б

Spark Streaming, Telegram и Kafka.

## Подготовка

Запускаем Kafka и выбираем ядро «ДЗ 3». Последняя ячейка сама запустит сборщик Telegram.

In [ ]:
import json
import re
import socket
import sys
from datetime import datetime
from operator import add, itemgetter
from pathlib import Path
from threading import Event, Thread

from kafka import KafkaConsumer
from pyspark import SparkConf, SparkContext
from pyspark.streaming import StreamingContext
from telegram_producer import start_producer

conf = SparkConf().setMaster("local[2]").setAppName("HW3-Streaming")
sc = SparkContext.getOrCreate(conf=conf)
sc.pythonExec = sys.executable
ssc = StreamingContext(sc, 30)

run_name = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
output = Path("output").resolve()
names_output = output / "names" / run_name
channels_output = output / "channels" / run_name
ssc.checkpoint(str(output / "checkpoint" / run_name))

## Поток из Kafka

Читаем новые сообщения из `telegram_messages` и передаём их в Spark через TCP. JSON разбираем один раз для всех заданий.

In [ ]:
consumer = KafkaConsumer(
    "telegram_messages", bootstrap_servers="localhost:9092",
    auto_offset_reset="latest", enable_auto_commit=False,
)
server = socket.socket()
server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
server.bind(("127.0.0.1", 9996))
server.listen(1)
stopped = Event()


def forward_kafka():
    try:
        connection, _ = server.accept()
        with connection:
            while not stopped.is_set():
                records = consumer.poll(timeout_ms=1000)
                for batch in records.values():
                    for record in batch:
                        connection.sendall(record.value + b"\n")
    except (BrokenPipeError, ConnectionResetError):
        pass
    finally:
        consumer.close()
        server.close()


worker = Thread(target=forward_kafka, daemon=True)
messages = ssc.socketTextStream("127.0.0.1", 9996).map(json.loads).cache()

## Задание 1. Подсчёт сообщений

Каждые 30 секунд показываем число сообщений за последнюю минуту и за последние 10 минут по убыванию. Окна считаем по времени поступления в Spark.

У анонимных публикаций отправителем может быть канал. Если у отправителя нет username, выводим его ID.

In [ ]:
def parse_message(message):
    return message["username"], 1


def message_count(item):
    return item[1]


def sort_counts(rdd):
    return rdd.sortBy(message_count, ascending=False, numPartitions=1)


def show_minute(time, rdd):
    print(f"\n{time} — окно 1 минута")
    for username, count in rdd.collect():
        print(f"{username}\t{count}")


def show_ten_minutes(time, rdd):
    print(f"\n{time} — окно 10 минут")
    for username, count in rdd.collect():
        print(f"{username}\t{count}")


batch_counts = messages.map(parse_message).reduceByKey(add)
minute_counts = batch_counts.window(60, 30).reduceByKey(add).transform(sort_counts)
ten_minute_counts = batch_counts.window(600, 30).reduceByKey(add).transform(sort_counts)
minute_counts.foreachRDD(show_minute)
ten_minute_counts.foreachRDD(show_ten_minutes)

## Задание 2. Частоты собственных имён

Берём русские слова с заглавной буквы, включая первое слово предложения. Приводим к нижнему регистру и убираем с конца гласные и «й». «Москва» и «Москвой» дают «москв».

In [ ]:
def extract_names(message):
    for word in re.findall(r"\b[А-ЯЁ][А-ЯЁа-яё]*\b", message["text"]):
        name = word.lower().rstrip("аеёиоуыэюяй")
        if name:
            yield (message["channel_id"], name), 1


def word_count(item):
    (channel_id, word), count = item
    return word, count


def update_count(values, previous):
    return (previous or 0) + sum(values)


def format_name(item):
    word, count = item
    return f"{word}, {count}"


top_words = []


def save_names(time, rdd):
    global top_words
    ordered = rdd.sortBy(itemgetter(1), ascending=False, numPartitions=1).cache()
    ordered.map(format_name).saveAsTextFile(str(names_output / time.strftime("%Y%m%d_%H%M%S")))
    top_words = ordered.take(10)
    ordered.unpersist()


minute_names = messages.flatMap(extract_names).window(60, 60).reduceByKey(add).cache()
name_counts = minute_names.map(word_count).reduceByKey(add).updateStateByKey(update_count)
name_counts.foreachRDD(save_names)

Раз в минуту сохраняем накопленные частоты по убыванию в `output/names/<запуск>/<минута>/part-00000`.

## Задание 3. Частоты имён по каналам

Считаем частоты отдельно для каждого канала. Минутные результаты лежат в `output/channels/<запуск>/<минута>/part-00000`.

In [ ]:
def sort_key(item):
    (channel_id, word), count = item
    return channel_id, -count


def format_channel_count(item):
    (channel_id, word), count = item
    return f"{channel_id}, {word}, {count}"


def channel_words(item):
    (channel_id, word), count = item
    return channel_id, [(word, count)]


def merge_top(first, second):
    return sorted(first + second, key=itemgetter(1), reverse=True)[:10]


top_by_channel = []


def save_channels(time, rdd):
    global top_by_channel
    ordered = rdd.sortBy(sort_key, numPartitions=1).cache()
    ordered.map(format_channel_count).saveAsTextFile(str(channels_output / time.strftime("%Y%m%d_%H%M%S")))
    top_by_channel = ordered.map(channel_words).reduceByKey(merge_top).collect()
    ordered.unpersist()


channel_counts = minute_names.updateStateByKey(update_count)
channel_counts.foreachRDD(save_channels)

## Общий запуск на 30 минут

Нажимаем Run All. Таблицы задания 1 появляются в последней ячейке каждые 30 секунд. После остановки там же выводятся топ-10 заданий 2 и 3.

Для досрочной остановки нажми Interrupt. Сборщик останется запущенным. Перед повторным Run All перезапусти ядро.

In [ ]:
worker.start()
ssc.start()
try:
    start_producer()
    finish_at = datetime.now().timestamp() + 30 * 60
    while datetime.now().timestamp() < finish_at:
        if ssc.awaitTerminationOrTimeout(1):
            break
except KeyboardInterrupt:
    pass
finally:
    stopped.set()
    ssc.stop(stopSparkContext=False, stopGraceFully=True)

print("\nЗадание 2 — топ-10")
print("Слово, количество")
for word, count in top_words:
    print(f"{word}, {count}")

print("\nЗадание 3 — топ-10 по каналам")
for channel_id, words in sorted(top_by_channel):
    print(f"\nКанал {channel_id}")
    print("Слово, количество")
    for word, count in words:
        print(f"{word}, {count}")